# Lab 01 - Autoloader 
#### Overview
Learn how Auto Loader performs incremental ingestion
using CloudFiles while understanding why each feature exists.

## Learning Objectives
By the end of this lab I should be able to explain:

✅ How Auto Loader discovers files

✅ How checkpoints provide idempotency

✅ Why duplicate rows can still occur

✅ How schema evolution works

✅ What rescued data is

✅ How schema hints work

✅ Production best practices

### Environment Setup
- Imports
- Config
- Catalog

In [0]:
%sql
USE CATALOG engineering_labs;
USE SCHEMA 01_bronze;

In [0]:
import sys

# Derive the workspace path from the current user so this cell is portable
_current_user = spark.sql("SELECT current_user()").collect()[0][0]
_repo_path = f"/Workspace/Users/{_current_user}/databricks-engineering-labs"
sys.path.insert(0, _repo_path)
from config.settings import *  # type: ignore[import-not-found]

## Reset Lab

In [0]:
def reset_lab(table_name: str, checkpoint: str):
    spark.sql(f"DROP TABLE IF EXISTS {BRONZE_SCHEMA}.{table_name}")
    print(f"Table {table_name} is dropped.")

    dbutils.fs.rm(f"{CHECKPOINT_PATH}/{checkpoint}", recurse=True)
    dbutils.fs.rm(SCHEMA_PATH, recurse=True)

    dbutils.fs.mkdirs(CHECKPOINT_PATH)
    dbutils.fs.mkdirs(SCHEMA_PATH)
    print("Checkpoint and schema subfolders created successfully.")

In [0]:
reset_lab(
    table_name="patients",
    checkpoint="patients"
)

### utils.autoloader 

In [0]:
def run_autoloader(
    file_name: str,
    table_name: str,
    checkpoint: str,
    file_format: str = "csv",
    reader_options: dict | None = None,
    writer_options: dict | None = None
):
    # Reader
    reader = (
            spark.readStream
                .format("cloudFiles")
                .option("cloudFiles.format", file_format)
                .option("header", "true")
                .option("rescuedDataColumn", "_rescued_data")
                .option("cloudFiles.schemaLocation", 
                        f"{SCHEMA_PATH}/{table_name}"
                        )
                .option("cloudFiles.inferColumnTypes", "true")
                .option("pathGlobFilter", file_name)
    )

    # Read Stream
    if reader_options:
        for key, value in reader_options.items():
            reader = reader.option(key, value)

    df = reader.load(RAW_PATH)

    # Write to stream
    writer = (
        df.writeStream
            .trigger(availableNow=True)
            .option(
                "checkpointLocation",
                f"{CHECKPOINT_PATH}/{checkpoint}"
            )
    )

    if writer_options:
        for key, value in writer_options.items():
            writer = writer.option(key, value)

    query = writer.toTable(f"{BRONZE_SCHEMA}.{table_name}")

    query.awaitTermination()

## Challenge 01 - Initial Ingestion
- cloudFiles
- schemaLocation
- checkpoints
- availableNow

**Expected outcome**:

- Auto Loader infers schema.
- Creates checkpoint.
- Loads 1163 rows.

In [0]:
run_autoloader(
    file_name="patients.csv",
    table_name="patients",
    checkpoint="patients",
    file_format="csv"
)

In [0]:
df_patients = spark.sql("""SELECT count(*) FROM `01_bronze`.patients;    
                        """).show()

### Inspect Auto Loader Metadata
checkpoints/
- patients/
    - commits/
    - offsets/
    - sources/
    - metadata/

In [0]:
display(dbutils.fs.ls(f"{SCHEMA_PATH}/patients/_schemas"))

In [0]:
display(dbutils.fs.ls(f"{CHECKPOINT_PATH}/patients"))

In [0]:
print(
    dbutils.fs.head(
        f"{SCHEMA_PATH}/patients/_schemas/0"
    )
)

In [0]:
dbutils.fs.head(
    f"{CHECKPOINT_PATH}/patients/commits/0"
)

In [0]:
display(
    dbutils.fs.ls(
        f"{CHECKPOINT_PATH}/patients/offsets"
    )
)

In [0]:
print(
    dbutils.fs.head(
        f"{CHECKPOINT_PATH}/patients/offsets/0"
    )
)

In [0]:
print(
    dbutils.fs.head(
        f"{CHECKPOINT_PATH}/patients/sources/0/rocksdb/logs/000004-3d55656d-2688-42e7-83e1-f818cf93b95a.log"
    )
)

## Challenge 02 - File Idempotency
Demonstrates that Auto Loader performs idempotent ingestion by tracking
processed files through checkpoints.

**Expected behavior:**
- patients.csv is processed once.
- Re-running the same ingestion does not duplicate data.
- patients_new.csv is considered a new file and is ingested once.
- Re-running patients_new.csv does not ingest it again because its path is already recorded in the checkpoint.

**Engineering concept:**

Auto Loader guarantees file-level idempotency, not record-level deduplication.

In [0]:
run_autoloader(
    file_name="patients_new.csv",
    table_name="patients",
    checkpoint="patients",
    file_format="csv"
)

In [0]:
incremental_ingestion_df = spark.sql("""
                                     SELECT COUNT(*) FROM `01_bronze`.patients
                                     """).show()

## Challenge 03 - Schema Evolution

Set `cloudFiles.schemaEvolutionMode` options:

- Default behavior (addNewColumns)
- Fail On New Columns
- Rescue Mode
- Type Widening
- None

In [0]:
run_autoloader(
    file_name="patients_schema_change.csv",
    table_name="patients",
    checkpoint="patients",
    file_format="csv",
    writer_options={
        "mergeSchema": "true"
    }
)

**Observation**

The default `addNewColumns` mode intentionally stops the stream when a new
column is detected.

On the first execution:

- Schema metadata is updated.
- The stream terminates.
- `automatic_retry=True`

Result:

- Auto Loader recognizes the updated schema.
- The DataFrame is created successfully.
- Delta Lake applies the schema evolution (`mergeSchema=true`).

In [0]:
df_schema_change = spark.read.table("patients")
df_schema_change.printSchema()

**Observation**

The `failOnNewColumns` mode intentionally stops the stream when a new
column is detected, and do not evolve schema.

**Expected**:
- Detect unknown column.
- Stop ingestion.
- Do not evolve schema.
- Require manual intervention.

**Result**:
- Stream failed.
- Unknown field detected.
- `automatic retry: false`
- Schema metadata was not updated.

In [0]:
run_autoloader(
    file_name="patients_schema_change.csv",
    table_name="patients",
    checkpoint="patients",
    reader_options={
        "cloudFiles.schemaEvolutionMode": "failOnNewColumns"
    }
)

In [0]:
run_autoloader(
    file_name="patients_type_widening.csv",
    table_name="patients",
    checkpoint="patients",
    reader_options={
        "cloudFiles.schemaEvolutionMode": "addNewColumnsWithTypeWidening"
    }
)

In [0]:
df_type_widening = spark.read.table("patients")
df_type_widening.printSchema()

In [0]:
run_autoloader(
    file_name="patients_schema_change.csv",
    table_name="patients",
    checkpoint="patients",
    reader_options={
        "cloudFiles.schemaEvolutionMode": "None"
    }
)

## Challenge 04 - Rescued Data

In [0]:
run_autoloader(
    file_name="patients_schema_change.csv",
    table_name="patients",
    checkpoint="patients",
    reader_options={
        "cloudFiles.schemaEvolutionMode": "rescue"
    }
)

**Observation**

Schema `rescue` preserves unexpected columns inside `_rescued_data`, but it does not update or merge existing records. Record deduplication and reconciliation belong in the Silver layer.

**Expected**:
- Preserve unknown columns inside `_rescued_data`.
- Do not evolve the table schema.
- Continue ingestion without failing.
- Preserve all incoming records.

**Result**:
- Stream completed successfully.
- `HEALTHCARE_PROVIDER` was stored in `_rescued_data`.
- The table schema remained unchanged.
- 1,164 rows contained non-null `_rescued_data`.
- Existing patient records were not updated or merged; a new version of each record was appended.

**Takeaway:***
`rescue` prioritizes data preservation over schema evolution. Instead of rejecting unexpected fields or modifying the table schema, Auto Loader safely stores unknown attributes in `_rescued_data`, allowing downstream pipelines to decide how and when those fields should be incorporated.

In [0]:
df_schema_change = spark.sql(""" 
                            SELECT id, _rescued_data
                            FROM patients
                            WHERE _rescued_data IS NOT NULL
                            LIMIT 5;
                             """).show(truncate=False)

In [0]:
df_schema_count = spark.sql("""
                            SELECT
                            COUNT(*) AS rescued_rows
                            FROM patients
                            WHERE _rescued_data IS NOT NULL;
                            """).show()

## Challenge 05 - Schema Hints

**Observation**

`cloudFiles.schemaHints` is only applied during the initial schema inference.

Once Auto Loader has persisted the inferred schema in `schemaLocation`, subsequent executions reuse that schema and ignore new schema hints.

**Expected:**
- Apply the specified data types during schema inference.
- Override automatic type inference for the specified columns.
- Persist the inferred schema for future ingestions.

**Result:**
- Schema hints were successfully applied after resetting the lab.
- `ZIP` was inferred as `STRING`.
- `BIRTHDATE` was inferred as `STRING`.
- The inferred schema was stored in `schemaLocation`.
- Future ingestions reused the stored schema instead of inferring it again.

**Takeaway**

Schema hints are not schema migration tools. They influence only the initial schema inference. If a schema has already been persisted in `schemaLocation`, the stored schema takes precedence over new hints. To test different schema hints, the schema metadata must be reset or a new `schemaLocation` must be used.

In [0]:
run_autoloader(
    file_name="patients_schema_hints.csv",
    table_name="patients",
    checkpoint="patients",
    reader_options={
        "cloudFiles.schemaHints": "ZIP STRING, BIRTHDATE STRING"
    }
)

In [0]:
df_schema_hints = spark.read.table("patients")
df_schema_hints.printSchema()

### Production Considerations

- Never store checkpoints inside DBFS root.

- Store checkpoints separately from raw data.

- One checkpoint per stream.

- Bronze should preserve raw records.

- Deduplication belongs in Silver.

- Auto Loader tracks files, not records.

- Schema evolution should be intentional.

###  Takeaways
- ✓ Auto Loader watches directories, not files.
- ✓ availableNow terminates the stream.
- ✓ schemaLocation stores inferred schemas.
- ✓ checkpointLocation stores ingestion progress.
- ✓ DBFS is disabled in Free Edition.
- ✓ Unity Catalog Volumes require /Volumes/catalog/schema/volume.

## References
- [What is Auto Loader? (Databricks Docs)](https://docs.databricks.com/aws/en/ingestion/cloud-object-storage/auto-loader/)
- [How Does Auto Loader Schema Evolution Works (Databricks Docs)](https://docs.databricks.com/aws/en/ingestion/cloud-object-storage/auto-loader/schema#how-does-auto-loader-schema-evolution-work)